<a href="https://colab.research.google.com/github/riyap1404/Be_Practical_Training/blob/main/Day10_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & API AUTHENTICATION
# Run this cell first to install required libraries and authenticate.
# ==============================================================================
!pip install -q -U google-genai chromadb pypdf fpdf langchain-text-splitters numpy

import os
import getpass
import numpy as np
import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from pypdf import PdfReader
from fpdf import FPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Official Google GenAI SDK
from google import genai
from google.genai import types

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion
# ------------------------------------------------------------------------------
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Google Gemini API Client initialized successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 785.1 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━

In [ ]:
# ==============================================================================
# CELL 1: GENERATE SYNTHETIC KNOWLEDGE BASE (PDF)
# Run this cell to create a dummy HR Policy PDF for our live ingestion demo.
# ==============================================================================
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)

document_text = """Acme Corp Global Employee Handbook (2026)

1. Remote Work Policy:
Employees may work remotely up to 3 days per week. Tuesdays and Thursdays are designated 'Anchor Days' where in-office presence is mandatory for collaborative meetings. Core working hours are 10:00 AM to 3:00 PM local time.

2. Annual Leave and PTO:
Full-time employees accrue 20 days of Paid Time Off (PTO) annually. Up to 5 unused PTO days can be rolled over to the next calendar year. Rollover days expire on March 31st.

3. Cloud Infrastructure Security:
All database access requires VPN connection and Multi-Factor Authentication (MFA). Database passwords must be rotated every 60 days. API keys must never be hardcoded into Git repositories.

4. Expense Reimbursements:
Business travel expenses must be submitted within 30 days of the trip via the Concur portal. Meal allowances are capped at $75 per diem. Alcohol is not a reimbursable expense.

5. Hardware Upgrades:
Engineers are eligible for a laptop refresh every 3 years. The standard issue is a 16-inch M-series MacBook Pro. Monitor allowance for home offices is $500.
"""

for line in document_text.split('\n'):
    pdf.cell(200, 10, txt=line, ln=True, align='L')

pdf_filename = "Acme_HR_Policy_2026.pdf"
pdf.output(pdf_filename)
print(f"✅ Generated synthetic PDF document: '{pdf_filename}'")

✅ Generated synthetic PDF document: 'Acme_HR_Policy_2026.pdf'


In [ ]:
# ==============================================================================
# SECTION 1: EMBEDDINGS & SIMILARITY METRICS
# ==============================================================================
"""
1. EMBEDDINGS FOR RETRIEVAL:
   - An embedding converts raw unstructured text into a dense array of floating-point numbers (e.g., 768 dimensions for Gemini text-embedding-004).
   - In this high-dimensional space, semantically similar sentences are physically closer together.
   - Example: "The dog barked" and "A puppy yelped" have no overlapping keywords, but their embeddings will have high similarity.

2. SIMILARITY METRICS:
   - Cosine Similarity: Measures the angle between two vectors (1.0 = identical direction, 0.0 = orthogonal).
     Standard for text because it ignores vector magnitude (document length).
   - Dot Product: Computes angle AND magnitude. If vectors are normalized (L2=1), Dot Product == Cosine Similarity.
   - Euclidean Distance (L2): Measures straight-line distance. Sensitive to document length.
"""

def compute_cosine_similarity(vec_a, vec_b):
    dot = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot / (norm_a * norm_b)

# Fetch embeddings for a query and two documents
resp = client.models.embed_content(
    model="gemini-embedding-001",
    contents=["Refund my money", "I want my cash back", "The server is down"]
)
vectors = [emb.values for emb in resp.embeddings]

sim_1 = compute_cosine_similarity(vectors[0], vectors[1])
sim_2 = compute_cosine_similarity(vectors[0], vectors[2])

print("=== 🧮 COSINE SIMILARITY DEMONSTRATION ===")
print(f"Query: 'Refund my money'")
print(f"Similarity to 'I want my cash back': {sim_1:.4f} (High Semantic Affinity)")
print(f"Similarity to 'The server is down':  {sim_2:.4f} (Low Semantic Affinity)")

=== 🧮 COSINE SIMILARITY DEMONSTRATION ===
Query: 'Refund my money'
Similarity to 'I want my cash back': 0.7256 (High Semantic Affinity)
Similarity to 'The server is down':  0.5584 (Low Semantic Affinity)


In [ ]:
# ==============================================================================
# SECTION 2: VECTOR DATABASES & CHUNKING STRATEGIES
# ==============================================================================
"""
VECTOR DATABASES (Why PostgreSQL isn't enough):
- Standard databases execute exact-match queries (WHERE text = 'x').
- Vector DBs (Chroma, FAISS, Pinecone) execute Approximate Nearest Neighbor (ANN) searches (e.g., HNSW indexing).
- They find the "closest" vectors in milliseconds across millions of documents.

CHUNKING STRATEGIES (The #1 reason RAG fails is bad chunking!):
1. Fixed-Size Chunking: Slice text every N tokens. (Flaw: Cuts sentences in half, destroying context).
2. Recursive Character Splitter: Tries to split on paragraphs (\n\n), then sentences (.), then words ( ), ensuring semantic blocks stay intact.
3. Overlap: Always overlap chunks by 10-20% to prevent losing context across boundaries.
"""

# Let's define our Custom Gemini Embedding Function for ChromaDB
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=input
        )
        # Return a list of float arrays
        return [emb.values for emb in response.embeddings]

gemini_ef = GeminiEmbeddingFunction()
print("✅ Custom Gemini Embedding Function for ChromaDB registered.")

✅ Custom Gemini Embedding Function for ChromaDB registered.


/tmp/ipykernel_528/2595921960.py:26: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  gemini_ef = GeminiEmbeddingFunction()


In [ ]:
# ==============================================================================
# SECTION 3: LIVE DEMO — END-TO-END DOCUMENT INGESTION PIPELINE
# ==============================================================================
# Step 1: Extract Text from PDF
reader = PdfReader("Acme_HR_Policy_2026.pdf")
raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() + "\n"

print("1. PDF Loaded & Parsed.")

# Step 2: Recursive Character Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # Max characters per chunk
    chunk_overlap=40,    # Context overlap
    separators=["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_text(raw_text)
print(f"2. Document Split into {len(chunks)} contextual chunks.")

# Step 3: Initialize ChromaDB and Ingest Data
chroma_client = chromadb.Client()

# Create or reset the collection
collection_name = "acme_hr_policy"
try:
    chroma_client.delete_collection(collection_name)
except:
    pass
collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=gemini_ef
)

# Prepare lists for Chroma
documents = []
ids = []
metadatas = []

for i, chunk in enumerate(chunks):
    documents.append(chunk)
    ids.append(f"chunk_{i}")
    metadatas.append({"source": "Acme_HR_Policy_2026.pdf", "chunk_index": i})

# Add to Vector Database (This automatically calls the Gemini Embedding API)
print("3. Embedding chunks and indexing in ChromaDB... (Please wait ~3 seconds)")
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)
print("✅ Ingestion Complete! Vector DB is ready for queries.")

1. PDF Loaded & Parsed.
2. Document Split into 9 contextual chunks.
3. Embedding chunks and indexing in ChromaDB... (Please wait ~3 seconds)
✅ Ingestion Complete! Vector DB is ready for queries.


In [ ]:
# ==============================================================================
# SECTION 4: LIVE DEMO — RETRIEVAL & SEMANTIC SEARCH
# ==============================================================================
# Now we act as the user asking a question
user_query = "What happens to my unused vacation days at the end of the year?"

print(f"👤 USER QUERY: '{user_query}'\n")

# Query ChromaDB (Embeds the query, does cosine similarity search, returns Top K)
results = collection.query(
    query_texts=[user_query],
    n_results=2 # Top-k retrieval
)

print("=== 🔍 VECTOR DB RETRIEVAL RESULTS ===")
for i, (doc, meta, distance) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0])):
    # Note: Chroma uses L2 distance by default. Lower is better/closer.
    print(f"\n--- Rank {i+1} (Distance: {distance:.4f}) ---")
    print(f"Source: {meta['source']} (Chunk {meta['chunk_index']})")
    print(f"Text Content:\n{doc}")

"""
THE NEXT STEP (RAG GENERATION):
In a full RAG pipeline, we take these retrieved chunks and inject them into the LLM prompt:
"Answer the user query using ONLY the following context: <chunk_1> <chunk_2>. Query: What happens..."
(We will cover the generation half deeply tomorrow!)
"""

👤 USER QUERY: 'What happens to my unused vacation days at the end of the year?'

=== 🔍 VECTOR DB RETRIEVAL RESULTS ===

--- Rank 1 (Distance: 0.5856) ---
Source: Acme_HR_Policy_2026.pdf (Chunk 3)
Text Content:
2. Annual Leave and PTO:
Full-time employees accrue 20 days of Paid Time Off (PTO) annually. Up to 5 unused PTO days can be rolled over to the next calendar year. Rollover days expire on March 31st.

--- Rank 2 (Distance: 0.9046) ---
Source: Acme_HR_Policy_2026.pdf (Chunk 8)
Text Content:
5. Hardware Upgrades:
Engineers are eligible for a laptop refresh every 3 years. The standard issue is a 16-inch M-series MacBook Pro. Monitor allowance for home offices is $500.


'\nTHE NEXT STEP (RAG GENERATION):\nIn a full RAG pipeline, we take these retrieved chunks and inject them into the LLM prompt:\n"Answer the user query using ONLY the following context: <chunk_1> <chunk_2>. Query: What happens..."\n(We will cover the generation half deeply tomorrow!)\n'

In [ ]:
# ==============================================================================
# SECTION 5: RAG VS. FINE-TUNING DECISION FRAMEWORK
# ==============================================================================
"""
THE MOST COMMON INDUSTRY CONFUSION: "Should I Fine-Tune or use RAG?"

┌───────────────────────┬──────────────────────────────────┬─────────────────────────────────┐
│ Feature               │ RAG (Retrieval-Augmented)        │ Fine-Tuning (SFT)               │
├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│ Primary Goal          │ Injecting specific, factual data │ Changing model tone, format, or │
│                       │ and reducing hallucinations.     │ teaching a new domain language. │
├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│ Data Freshness        │ Real-time. Update the Vector DB  │ Static. Model weights must be   │
│                       │ and the model knows it instantly.│ retrained to learn new facts.   │
├───────────────────────┼──────────────────────┼─────────────────────────────────────────────┤
│ Hallucination Control │ High. Grounded by strict prompt  │ Low/Medium. Prone to mixing up  │
│                       │ constraints and citations.       │ facts embedded in weights.      │
├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│ Cost & Speed to Build │ Fast & Cheap (Days/Weeks).       │ Slow & Expensive (Weeks/Months).│
└───────────────────────┴──────────────────────────────────┴─────────────────────────────────┘

RULE OF THUMB:
1. If the model needs to know NEW FACTS -> Use RAG.
2. If the model needs to learn a NEW SKILL or TONE -> Use Fine-Tuning.
3. If both -> Do Both (Fine-Tune for format, RAG for facts).
"""

'\nTHE MOST COMMON INDUSTRY CONFUSION: "Should I Fine-Tune or use RAG?"\n\n┌───────────────────────┬──────────────────────────────────┬─────────────────────────────────┐\n│ Feature               │ RAG (Retrieval-Augmented)        │ Fine-Tuning (SFT)               │\n├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤\n│ Primary Goal          │ Injecting specific, factual data │ Changing model tone, format, or │\n│                       │ and reducing hallucinations.     │ teaching a new domain language. │\n├───────────────────────┼──────────────────────────────────┼─────────────────────────────────┤\n│ Data Freshness        │ Real-time. Update the Vector DB  │ Static. Model weights must be   │\n│                       │ and the model knows it instantly.│ retrained to learn new facts.   │\n├───────────────────────┼──────────────────────┼─────────────────────────────────────────────┤\n│ Hallucination Control │ High. Grounded by strict prompt  │ 

In [ ]:
# ==============================================================================
# SECTION 6: STUDENT GRADED LAB — INGESTION & RETRIEVAL
# ==============================================================================
"""
🎓 STUDENT LAB INSTRUCTIONS:
You are building the retrieval engine for an IT Support Knowledge Base.

MANDATORY TASKS:
1. Review the provided string variable `it_knowledge_base`.
2. Use the LangChain `RecursiveCharacterTextSplitter` to chunk the text.
   - Set chunk_size = 150
   - Set chunk_overlap = 25
3. Create a NEW ChromaDB collection named "it_support_kb".
4. Ingest your chunks, ids, and metadata into the collection.
5. Run the 3 test queries below and print the top 1 result for each.
"""

it_knowledge_base = """
IT Support Troubleshooting Guide:

1. VPN Connection Failures:
If Cisco AnyConnect fails with 'Connection Timeout', ensure the user is not on a public hotel Wi-Fi network that blocks UDP port 443. Instruct the user to switch to a mobile hotspot.

2. Printer Installation:
To map the 3rd floor marketing printer, users must connect to the corporate network, open File Explorer, type '\\printserver01\MKTG_Color_Laser', and double-click to install drivers automatically.
"""

# ------------------------------------------------------------------------------
# STUDENT WORKSPACE (WRITE YOUR SOLUTION BELOW)
# ------------------------------------------------------------------------------

# 1. Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=25,
    separators=["\n\n", "\n", ".", " "]
)
it_chunks = text_splitter.split_text(it_knowledge_base)
print(f"Document Split into {len(it_chunks)} contextual chunks.")

# 2. Database Creation & Ingestion
it_collection_name = "it_support_kb"
it_chroma_client = chromadb.Client()

try:
    it_chroma_client.delete_collection(it_collection_name)
except:
    pass

it_collection = it_chroma_client.create_collection(
    name=it_collection_name,
    embedding_function=gemini_ef
)

# Prepare lists for Chroma
it_documents = []
it_ids = []
it_metadatas = []

for i, chunk in enumerate(it_chunks):
    it_documents.append(chunk)
    it_ids.append(f"it_chunk_{i}")
    it_metadatas.append({"source": "IT_Knowledge_Base", "chunk_index": i})

# 3. Add to Collection
print("Embedding chunks and indexing in 'it_support_kb' ChromaDB... (Please wait)")
it_collection.add(
    documents=it_documents,
    ids=it_ids,
    metadatas=it_metadatas
)
print("✅ IT Support KB Ingestion Complete!")

# 4. Test Queries
test_queries = [
    "VPN is not connecting on hotel wifi.",
    "How do I install the marketing printer?",
    "My account is locked, what do I do?"
]

print("\n=== 🔍 IT SUPPORT KB TEST QUERIES ===")
for query in test_queries:
    print(f"\n👤 USER QUERY: '{query}'")
    results = it_collection.query(
        query_texts=[query],
        n_results=1
    )
    if results["documents"] and results["documents"][0]:
        doc = results["documents"][0][0]
        meta = results["metadatas"][0][0]
        distance = results["distances"][0][0]
        print(f"--- Top 1 Result (Distance: {distance:.4f}) ---")
        print(f"Source: {meta['source']} (Chunk {meta['chunk_index']})")
        print(f"Text Content:\n{doc}")
    else:
        print("No relevant results found.")

<>:25: SyntaxWarning: invalid escape sequence '\M'
<>:25: SyntaxWarning: invalid escape sequence '\M'
/tmp/ipykernel_528/400660740.py:25: SyntaxWarning: invalid escape sequence '\M'
  To map the 3rd floor marketing printer, users must connect to the corporate network, open File Explorer, type '\\printserver01\MKTG_Color_Laser', and double-click to install drivers automatically.


Document Split into 8 contextual chunks.
Embedding chunks and indexing in 'it_support_kb' ChromaDB... (Please wait)
✅ IT Support KB Ingestion Complete!

=== 🔍 IT SUPPORT KB TEST QUERIES ===

👤 USER QUERY: 'VPN is not connecting on hotel wifi.'
--- Top 1 Result (Distance: 0.5383) ---
Source: IT_Knowledge_Base (Chunk 2)
Text Content:
If Cisco AnyConnect fails with 'Connection Timeout', ensure the user is not on a public hotel Wi-Fi network that blocks UDP port 443

👤 USER QUERY: 'How do I install the marketing printer?'
--- Top 1 Result (Distance: 0.4453) ---
Source: IT_Knowledge_Base (Chunk 4)
Text Content:
2. Printer Installation:

👤 USER QUERY: 'My account is locked, what do I do?'
--- Top 1 Result (Distance: 0.8766) ---
Source: IT_Knowledge_Base (Chunk 3)
Text Content:
. Instruct the user to switch to a mobile hotspot.
